# 03 MLP and TabNet

Train the deep learning models. TabNet is skipped gracefully if `pytorch-tabnet` is unavailable.


In [ ]:
# Colab setup. Run this cell if dependencies are missing.
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    # drive.mount('/content/drive')  # Optional: uncomment if storing project/data in Drive.
    !pip install -q -r requirements.txt

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print('Project root:', PROJECT_ROOT)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


In [ ]:
from src.data_download import download_dataset
from src.preprocessing import prepare_splits, fit_transform_deep
from src.train_deep import train_mlp, predict_mlp, train_tabnet, predict_tabnet
from src.evaluate import choose_threshold_max_f1, metrics_row, save_metrics
from src.plots import plot_roc_curve, plot_precision_recall_curve, plot_confusion_matrix
from src.config import FIGURES_DIR, MODELS_DIR
import joblib

csv_path = download_dataset()
split = prepare_splits(csv_path)
deep = fit_transform_deep(split)
joblib.dump(deep, MODELS_DIR / 'deep_preprocessing.joblib')
print(deep.X_cat_train.shape, deep.X_num_train.shape)


In [ ]:
rows = []
mlp, history = train_mlp(deep, epochs=30, patience=5)
mlp_val = predict_mlp(mlp, deep.X_cat_val, deep.X_num_val)
mlp_threshold = choose_threshold_max_f1(deep.y_val, mlp_val)
mlp_test = predict_mlp(mlp, deep.X_cat_test, deep.X_num_test)
rows.append(metrics_row('MLP', 'test', deep.y_test, mlp_test, mlp_threshold, 'val_max_f1'))
rows[-1]


In [ ]:
tabnet_test = None
try:
    tabnet = train_tabnet(deep)
    tabnet_val = predict_tabnet(tabnet, deep.X_cat_val, deep.X_num_val)
    tabnet_threshold = choose_threshold_max_f1(deep.y_val, tabnet_val)
    tabnet_test = predict_tabnet(tabnet, deep.X_cat_test, deep.X_num_test)
    rows.append(metrics_row('TabNet', 'test', deep.y_test, tabnet_test, tabnet_threshold, 'val_max_f1'))
except Exception as exc:
    print('TabNet failed or is unavailable:', exc)

rows


In [ ]:
save_metrics(rows, append=True)

plot_roc_curve(deep.y_test, mlp_test, 'MLP', FIGURES_DIR / 'mlp_roc.png')
plot_precision_recall_curve(deep.y_test, mlp_test, 'MLP', FIGURES_DIR / 'mlp_pr.png')
plot_confusion_matrix(deep.y_test, mlp_test, 'MLP', mlp_threshold, FIGURES_DIR / 'mlp_confusion_matrix.png')

if tabnet_test is not None:
    plot_roc_curve(deep.y_test, tabnet_test, 'TabNet', FIGURES_DIR / 'tabnet_roc.png')
    plot_precision_recall_curve(deep.y_test, tabnet_test, 'TabNet', FIGURES_DIR / 'tabnet_pr.png')
    plot_confusion_matrix(deep.y_test, tabnet_test, 'TabNet', tabnet_threshold, FIGURES_DIR / 'tabnet_confusion_matrix.png')
